# Satellite-to-Satellite Validation: SOCRATES

This notebook tests TAT-C's satellite-to-satellite analysis (`collect_space_observations`) on very brief windows: the close approaches between satellites, which pass within a few kilometers of each other at up to 15 km/s, so that they stay within a range limit of 10 km for as little as a second. The reference is CelesTrak's [SOCRATES](https://celestrak.org/SOCRATES/) (Satellite Orbital Conjunction Reports Assessing Threatening Encounters in Space), which lists the close approaches predicted from published two-line element sets (TLEs) over the following week, with their time and range of closest approach. SOCRATES and TAT-C both propagate TLEs with SGP4, so this is a comparison of two implementations from the same element sets, rather than with truth: it tests TAT-C's propagation and its search for windows at the extremes of brevity. Two companion notebooks validate the range and occlusion constraints with truth: [COSMIC-2](ValidateSpaceCOSMIC2.ipynb) (radio occultations) and [GRACE-FO](ValidateSpaceGRACEFO.ipynb) (precise orbits).

1. **Closest approach.** Do TAT-C's propagated orbits reproduce SOCRATES's times and ranges of closest approach?
2. **Brief windows.** Does `collect_space_observations` find every window at least `min_duration` long, as documented, and how many shorter ones?

## Reference Data

SOCRATES publishes only its current run (three runs a day). The run of 25 October 2026 (06:00 UTC) listed 195,380 close approaches within 5 km over the following week, each with the two objects, the time and range of closest approach (`TCA` and `TCA_RANGE`), their relative speed, and the days from each object's element set epoch to the closest approach (`DSE_1` and `DSE_2`, to 0.001 day), which identify the element sets used. CelesTrak's element sets of active satellites, downloaded the same day, match those epochs (within 60 s) for both objects of 79,845 approaches. A seeded random sample of 100 of them in each of five relative speed bands (0 to 0.1, 0.1 to 1, 1 to 5, 5 to 10, and 10 to 16 km/s), and the element sets of their 732 satellites, are included with this notebook (`socrates_2026-10-25.csv` and `socrates_2026-10-25.tle`, data from CelesTrak). If a newer SOCRATES run (`sort-minRange.csv`) and active element sets (`active.tle`, `gp.php?GROUP=active&FORMAT=tle`) are saved in a local `data/socrates` directory, the notebook draws a new sample from them.

The approaches of the slowest band are between satellites flying in formation or nearly the same orbit, such as Starlink satellites of the same orbital shell, whose range changes slowly and repeatedly around the orbit; the fastest are between satellites in crossing orbits.

In [1]:
from datetime import datetime, timedelta, timezone
from pathlib import Path

import numpy as np
import pandas as pd

SAMPLE_FILE = Path("socrates_2026-10-25.csv")
TLE_FILE = Path("socrates_2026-10-25.tle")
SNAPSHOT_DIR = (
    Path("data") / "socrates"
)  # optional: a fresh SOCRATES run and active element sets
BANDS = [-0.01, 0.1, 1, 5, 10, 16]  # km/s, relative speed bands
LABELS = [
    "0 to 0.1 km/s",
    "0.1 to 1 km/s",
    "1 to 5 km/s",
    "5 to 10 km/s",
    "10 to 16 km/s",
]


def epoch(line1):
    """Epoch of a TLE from its first line."""
    return datetime(2000 + int(line1[18:20]), 1, 1, tzinfo=timezone.utc) + timedelta(
        days=float(line1[20:32]) - 1
    )


if (SNAPSHOT_DIR / "sort-minRange.csv").exists() and (
    SNAPSHOT_DIR / "active.tle"
).exists():
    # resample from a SOCRATES run: approaches whose element sets (epochs, from the days since epoch at the
    # time of closest approach, to 0.001 day) are those in the active satellites' element sets
    run = pd.read_csv(SNAPSHOT_DIR / "sort-minRange.csv")
    lines = [
        line.rstrip() for line in open(SNAPSHOT_DIR / "active.tle") if line.strip()
    ]
    element_sets = {
        int(lines[i + 1][2:7]): lines[i : i + 3] for i in range(0, len(lines), 3)
    }
    tca = pd.to_datetime(run.TCA, utc=True)
    matched = np.ones(len(run), dtype=bool)
    for k in (1, 2):
        epochs = pd.to_datetime(
            run[f"NORAD_CAT_ID_{k}"].map(
                lambda n: epoch(element_sets[n][1]) if n in element_sets else None
            ),
            utc=True,
        )
        matched &= (
            (epochs - (tca - pd.to_timedelta(run[f"DSE_{k}"], unit="D")))
            .abs()
            .le(pd.Timedelta(seconds=60))
            .to_numpy()
        )
    print(f"{matched.sum()} of {len(run)} approaches with matching element sets")
    candidates = run[matched]
    sample = (
        candidates.groupby(pd.cut(candidates.TCA_RELATIVE_SPEED, BANDS), observed=True)
        .sample(100, random_state=20261025)
        .sort_values("TCA")
        .reset_index(drop=True)
    )
    sample.to_csv(SAMPLE_FILE, index=False)
    ids = sorted(set(sample.NORAD_CAT_ID_1) | set(sample.NORAD_CAT_ID_2))
    TLE_FILE.write_text("\n".join(line for n in ids for line in element_sets[n]) + "\n")
sample = pd.read_csv(SAMPLE_FILE)
sample["tca"] = pd.to_datetime(sample.TCA, utc=True)
sample["band"] = pd.cut(sample.TCA_RELATIVE_SPEED, BANDS, labels=LABELS)
sample["days since epoch"] = sample[["DSE_1", "DSE_2"]].max(axis=1)
lines = [line.rstrip() for line in open(TLE_FILE) if line.strip()]
tles = {int(lines[i + 1][2:7]): lines[i + 1 : i + 3] for i in range(0, len(lines), 3)}
print(
    f"{len(sample)} approaches of {len(tles)} satellites, {sample.tca.min():%Y-%m-%d %H:%M} to {sample.tca.max():%Y-%m-%d %H:%M}"
)
sample.groupby("band", observed=True).agg(
    approaches=("TCA", "size"),
    **{
        "minimum range, median (km)": ("TCA_RANGE", "median"),
        "relative speed, median (km/s)": ("TCA_RELATIVE_SPEED", "median"),
        "days since epoch, max": ("days since epoch", "max"),
    },
).round(3)

79845 of 195380 approaches with matching element sets
500 approaches of 732 satellites, 2026-10-25 06:16 to 2026-11-01 05:44


,approaches,"minimum range, median (km)","relative speed, median (km/s)","days since epoch, max"
band,,,,
0 to 0.1 km/s,100,1.950,0.003,10.693
0.1 to 1 km/s,100,3.528,0.442,10.901
1 to 5 km/s,100,3.305,3.012,8.505
5 to 10 km/s,100,3.525,7.635,8.350
10 to 16 km/s,100,3.647,14.212,10.699


## Test 1: Closest Approach

`collect_space_observations` reports the closest approach of the satellites in each window: its time (`closest_time`) and slant range (`closest_range`), the least range among the samples of its search refined to 0.1 ms. It is run for each approach over an hour centered on SOCRATES's time of closest approach, with a `max_range` of 10 km (within which every sampled approach passes), no occlusion constraint, and a `min_duration` of 1 s, and the closest approach of the window nearest SOCRATES's is compared with it. SOCRATES reports ranges to a meter and times to a millisecond.

In [2]:
from tatc.analysis import collect_space_observations
from tatc.constants import timescale
from tatc.schemas import GeneralPerturbationsOrbit, Satellite

satellites = {
    n: Satellite(name=str(n), orbit=GeneralPerturbationsOrbit.from_tle(tle))
    for n, tle in tles.items()
}


def slant_range(a, b, times):
    """Range (m) between two satellites at times (list of datetimes)."""
    t = timescale.from_datetimes(list(times))
    return np.linalg.norm(
        a.orbit.get_orbit_track_at_time(t).position.m
        - b.orbit.get_orbit_track_at_time(t).position.m,
        axis=0,
    )


LIMIT = 10e3  # m, max_range
HALF_WINDOW = timedelta(minutes=30)  # analysis period around each closest approach

rows = []
for approach in sample.itertuples():
    a, b = satellites[approach.NORAD_CAT_ID_1], satellites[approach.NORAD_CAT_ID_2]
    tca = approach.tca.to_pydatetime()
    windows = collect_space_observations(
        a,
        tca - HALF_WINDOW,
        tca + HALF_WINDOW,
        to_satellites=b,
        max_range=LIMIT,
        min_grazing_altitude=None,
        min_duration=timedelta(seconds=1),
    )
    # the window whose closest approach is nearest SOCRATES's
    nearest = windows.iloc[
        np.argmin(np.abs((windows.closest_time - approach.tca).dt.total_seconds()))
    ]
    rows.append({"tatc_tca": nearest.closest_time, "tatc_range": nearest.closest_range})
sample = pd.concat([sample, pd.DataFrame(rows)], axis=1)
sample["range difference (m)"] = sample.tatc_range - sample.TCA_RANGE * 1e3
sample["time difference (ms)"] = (
    pd.to_datetime(sample.tatc_tca, utc=True) - sample.tca
).dt.total_seconds() * 1e3
sample["starlink"] = sample.OBJECT_NAME_1.str.contains(
    "STARLINK"
) | sample.OBJECT_NAME_2.str.contains("STARLINK")
summary = sample.groupby("band", observed=True).agg(
    **{
        "|range difference|, median (m)": (
            "range difference (m)",
            lambda x: x.abs().median(),
        ),
        "95% (m)": ("range difference (m)", lambda x: x.abs().quantile(0.95)),
        "max (m)": ("range difference (m)", lambda x: x.abs().max()),
        "|time difference|, median (ms)": (
            "time difference (ms)",
            lambda x: x.abs().median(),
        ),
        "95% (ms)": ("time difference (ms)", lambda x: x.abs().quantile(0.95)),
    }
)
large = sample[sample["range difference (m)"].abs() > 10]
print(
    f"{len(large)} approaches differ by more than 10 m in range, {large.starlink.mean():.0%} involving Starlink "
    f"(all approaches: {sample.starlink.mean():.0%}), {large['days since epoch'].median():.1f} days after epoch "
    f"(median; all approaches: {sample['days since epoch'].median():.1f})"
)
summary.round(2)

22 approaches differ by more than 10 m in range, 91% involving Starlink (all approaches: 59%), 6.6 days after epoch (median; all approaches: 5.1)


,"|range difference|, median (m)",95% (m),max (m),"|time difference|, median (ms)",95% (ms)
band,,,,,
0 to 0.1 km/s,0.30,3.19,666.68,20.52,479.25
0.1 to 1 km/s,0.64,14.29,37.21,0.31,0.96
1 to 5 km/s,0.70,13.52,161.25,0.25,1.00
5 to 10 km/s,0.75,9.75,73.91,0.23,1.15
10 to 16 km/s,0.31,1.38,59.60,0.25,0.59


The closest approaches `collect_space_observations` reports agree with SOCRATES's. Ranges agree to within a meter (a median of 0.3 to 0.75 m by band, about the precision of SOCRATES's ranges) and within 14 m for 95% of approaches. Times agree within 0.3 ms (median) and about 1 ms (95%), except for satellites in nearly the same orbit (0 to 0.1 km/s), whose range changes so slowly near its minimum that its time is defined only to tens of milliseconds.

Twenty approaches (4%) differ by 13 to 161 m in range, almost all involving a Starlink satellite and further from their element sets' epochs (6.6 days, median, against 5.1 days for all approaches). Differences that grow with propagation from epoch, for element sets with large drag terms, are consistent with small differences between implementations of SGP4 (for example, in their options), and are well within the accuracy of the element sets themselves. One pair in nearly the same orbit stays within 10 km for the whole hour, so its window spans two of SOCRATES's approaches; `collect_space_observations` reports the least range within the window (2.4 km, at its start), not SOCRATES's 3.1 km.

## Test 2: Brief Windows

`collect_space_observations` is run for each approach over an hour centered on SOCRATES's time of closest approach, with a `max_range` of 10 km (within which every sampled approach passes), no occlusion constraint, and a `min_duration` of 60, 10, 1, or 0.1 s. The search samples the range at half of `min_duration` and refines each change between samples, so it is guaranteed to find windows at least `min_duration` long, but may miss shorter ones (falling between samples). The true window is found from finely sampled ranges around TAT-C's closest approach (Test 1), within the hour, and is found if a window contains that closest approach.

In [3]:
import time

DURATIONS = [60, 10, 1, 0.1]  # s, min_duration


def true_window(a, b, tca, speed, start, end):
    """Duration (s) of the window around the closest approach when the range is within LIMIT, within the analysis
    period (from finely sampled ranges)."""
    half = min(HALF_WINDOW.total_seconds(), 2 * LIMIT / max(speed * 1e3, 1e-3) + 5)
    offsets = np.linspace(-half, half, 40001)
    times = [tca + timedelta(seconds=float(s)) for s in offsets]
    inside = (slant_range(a, b, times) <= LIMIT) & np.array(
        [start <= t <= end for t in times]
    )
    k = len(offsets) // 2
    if not inside[k]:
        return 0.0
    lo = k - np.argmin(inside[k::-1]) + 1 if not inside[: k + 1].all() else 0
    hi = k + np.argmin(inside[k:]) if not inside[k:].all() else len(offsets)
    return offsets[hi - 1] - offsets[lo]


rows = []
for approach in sample.itertuples():
    a, b = satellites[approach.NORAD_CAT_ID_1], satellites[approach.NORAD_CAT_ID_2]
    tca = pd.Timestamp(approach.tatc_tca).floor("us").to_pydatetime()
    start, end = (
        approach.tca.to_pydatetime() - HALF_WINDOW,
        approach.tca.to_pydatetime() + HALF_WINDOW,
    )
    row = {
        "band": approach.band,
        "window (s)": true_window(a, b, tca, approach.TCA_RELATIVE_SPEED, start, end),
    }
    for duration in DURATIONS:
        tic = time.perf_counter()
        windows = collect_space_observations(
            a,
            start,
            end,
            to_satellites=b,
            max_range=LIMIT,
            min_grazing_altitude=None,
            min_duration=timedelta(seconds=duration),
        )
        row[f"time {duration} s"] = time.perf_counter() - tic
        row[f"found {duration} s"] = bool(
            (
                (windows.start <= pd.Timestamp(tca))
                & (windows.end >= pd.Timestamp(tca))
            ).any()
        )
    rows.append(row)
found = pd.DataFrame(rows)
found["band"] = pd.Categorical(found.band, categories=LABELS, ordered=True)
print(
    "true window within 10 km, median (s): "
    + ", ".join(
        f"{b} {w:.1f}"
        for b, w in found.groupby("band", observed=True)["window (s)"].median().items()
    )
    + f"; shortest {found['window (s)'].min():.2f} s"
)
pd.DataFrame(
    {
        f"min_duration = {d} s": {
            "windows at least min_duration": int((found["window (s)"] >= d).sum()),
            "found (%)": 100
            * found.loc[found["window (s)"] >= d, f"found {d} s"].mean(),
            "shorter windows": int((found["window (s)"] < d).sum()),
            "shorter found (%)": (
                100 * found.loc[found["window (s)"] < d, f"found {d} s"].mean()
                if (found["window (s)"] < d).any()
                else np.nan
            ),
            "time per hour analyzed, median (ms)": 1e3 * found[f"time {d} s"].median(),
        }
        for d in DURATIONS
    }
).T.round(2)

true window within 10 km, median (s): 0 to 0.1 km/s 3599.7, 0.1 to 1 km/s 42.4, 1 to 5 km/s 6.0, 5 to 10 km/s 2.4, 10 to 16 km/s 1.3; shortest 1.15 s


,windows at least min_duration,found (%),shorter windows,shorter found (%),"time per hour analyzed, median (ms)"
min_duration = 60 s,137.0,100.0,363.0,10.47,3.62
min_duration = 10 s,224.0,100.0,276.0,18.48,28.63
min_duration = 1 s,500.0,100.0,0.0,NaN,43.30
min_duration = 0.1 s,500.0,100.0,0.0,NaN,190.82


`collect_space_observations` finds every window at least `min_duration` long, for every `min_duration` from 60 s to 0.1 s, as documented, and 10 to 18% of the shorter windows, those its samples happen to fall in. The windows within 10 km last from 1.15 s (a median of 1.3 s at 10 to 16 km/s) to the whole hour (satellites in nearly the same orbit), so a `min_duration` of 1 s finds all of them. The cost grows as `min_duration` shrinks, since the search samples the whole analysis period at half of it: for a pair of satellites, a median of 4 ms per hour analyzed at 60 s, 43 ms at 1 s, and 191 ms at 0.1 s, including the refinement of each window's closest approach (about 20 ms per call).

## Summary

| Test | Result |
| --- | --- |
| 1. Closest approach | `closest_range` within 1 m (median) and 14 m (95%) of SOCRATES's ranges, `closest_time` within 0.3 ms (median), except for satellites in nearly the same orbit; 4% of approaches (almost all involving Starlink) differ by 13 to 161 m |
| 2. Brief windows | Every window at least `min_duration` long found, for `min_duration` from 60 to 0.1 s, and 10 to 18% of shorter windows; 43 ms per hour analyzed for a pair of satellites with a `min_duration` of 1 s, including the closest approach |

**TAT-C's propagation and window search hold for close approaches.** From the same element sets, `collect_space_observations` reproduces SOCRATES's closest approaches to about a meter and a millisecond, and finds every window within a range limit that lasts at least `min_duration`, down to windows of about a second at relative speeds up to 15 km/s.

**Closest approaches and cost.** `collect_space_observations` reports each window's closest approach (`closest_time` and `closest_range`), which reproduces SOCRATES's, except where a window spans several approaches, when it is the least of them. The cost of a fine `min_duration`, which samples the whole analysis period, grows quickly for many pairs of satellites over long periods: with a `min_duration` of 1 s, about 4 s per pair for a week.

**Next steps:**

1. *Debris.* Extend the sample to approaches involving debris, whose element sets are not in CelesTrak's active satellites (for example, from Space-Track's history of element sets).
2. *Sub-second windows.* Test smaller range limits, whose windows last a fraction of a second.

**Limitations.** This is a comparison of two implementations from the same element sets, not with truth, and covers active satellites only. With a range limit of 10 km, the shortest window lasts about a second.